# SO-101 Phase 6A — Google ColabでGitHubのコードを実行

このNotebookはGitHubをclone / pullし、**実機操作なし**でMuJoCoの動的姿勢保持テストを行います。描画は不要です。GPUなしのCPUランタイムで構いません。

元の失敗ログは初期自己接触12件・2〜4msで大きな姿勢ずれ。新しい`--self-collision off`は、物体のないPhase 6Aで自己衝突だけを切り、重力・慣性・PDを残す切り分け用です。把持成功の証明ではありません。


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'mujoco>=3.3,<4', 'pandas', 'matplotlib'],check=True)
import mujoco
print('MuJoCo',mujoco.__version__)


In [ ]:
from pathlib import Path
REPO=Path('/content/so101-imitation-learning')
REMOTE='https://github.com/neogim/so101-imitation-learning.git'
BRANCH='feature/so101-dynamic-arm-control'
if not REPO.exists():
    subprocess.run(['git','clone','--single-branch','--branch',BRANCH,REMOTE,str(REPO)],check=True)
else:
    current=subprocess.check_output(['git','-C',str(REPO),'branch','--show-current'],text=True).strip()
    assert current==BRANCH, f'Unexpected local branch: {current}'
    subprocess.run(['git','-C',str(REPO),'pull','--ff-only','origin',BRANCH],check=True)
print(subprocess.check_output(['git','-C',str(REPO),'log','-1','--oneline'],text=True))
assert (REPO/'data/urdf/so101/so101_new_calib.urdf').is_file()
assert (REPO/'data/urdf/so101/assets').is_dir()


In [ ]:
from datetime import datetime, timezone
import json
STAMP=datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S')
OUT=Path('/content/phase6-results')/f'phase6a-colab-{STAMP}'
cmd=[sys.executable,'-u',str(REPO/'scripts/so101_sim/phase6_dynamic_joint_hold.py'),'--duration-s','1','--self-collision','off','--out',str(OUT)]
print('Executing:', ' '.join(cmd))
subprocess.run(cmd,cwd=REPO,check=True)
summary=json.loads((OUT/'summary.json').read_text())
print('Initial contacts:',summary['initial_contacts_count'])
print('Maximum errors [deg]:',summary['max_abs_error_deg'])


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
df=pd.read_csv(OUT/'joint_trace.csv')
names=('shoulder_pan','shoulder_lift','elbow_flex','wrist_flex','wrist_roll')
ax=df.plot(x='time_s',y=[f'{n}_error_deg' for n in names],figsize=(10,4))
ax.set_xlabel('Time (s)'); ax.set_ylabel('Target minus actual joint angle (deg)')
ax.grid(True); plt.show()


## 結果をGoogle Driveに保存（任意）

出張中に結果を保存したいときだけこのセルを実行し、Google Driveへのアクセスを承認してください。保存対象はCSVとJSONだけです。公開GitHubへ校正データや実験ログをpushしません。


In [ ]:
from google.colab import drive
import shutil
drive.mount('/content/drive')
DEST=Path('/content/drive/MyDrive/SO101_Phase6/hold_results')/OUT.name
DEST.parent.mkdir(parents=True,exist_ok=True)
shutil.copytree(OUT,DEST)
print('Saved:',DEST)
